# Dynamic Flight Fare Prediction — Regression Project
## IIT Bombay Interview-Oriented End-to-End ML Project

**Goal:** Predict flight ticket price from airline, route, travel timing, stops, class, duration and days left before departure.

**Problem type:** Supervised **Regression**

**Target:** `price`

**Dataset source:** Kaggle — *Flight Price Prediction* by Shubham Bathwal. The dataset documentation lists `price` as the target and includes airline, flight, source city, departure time, stops, arrival time, destination city, class, duration and days left. It is tagged as a regression dataset. 

**Dataset:** https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction


## 1. Problem Statement

Airline ticket prices vary according to route, airline, travel timing, class, number of stops, duration and how many days remain before departure. The objective is to build a machine-learning regression system that estimates the expected ticket price from these observable factors.

### Interview-ready statement

> “I developed a regression-based flight fare prediction system to estimate ticket prices from flight characteristics and booking lead time. I compared linear and tree-based regression models, used leakage-safe preprocessing and feature engineering, evaluated them using MAE, RMSE and R², and used feature importance and residual analysis to understand model behavior.”

**Important limitation:** this dataset supports fare prediction, but it does not by itself prove a causal or real-time dynamic-pricing model. `days_left` is a booking-lead-time proxy; true dynamic pricing would ideally require repeated price observations over time for the same flight.


## 2. Project Architecture

```text
Flight Dataset
      ↓
Data Quality Check
      ↓
EDA
      ↓
Feature Engineering
      ↓
Train / Test Split
      ↓
Leakage-safe Preprocessing
      ├── Numerical → Imputation → Scaling
      └── Categorical → Imputation → One-Hot Encoding
      ↓
Baseline Regression
      ↓
Regularized Regression
      ├── Ridge
      └── Lasso
      ↓
Nonlinear Regression
      ├── Random Forest
      ├── Gradient Boosting
      └── HistGradientBoosting
      ↓
Cross Validation + Hyperparameter Tuning
      ↓
MAE / RMSE / R² / MAPE
      ↓
Residual Analysis
      ↓
Feature Importance
      ↓
Final Fare Prediction
```


## 3. Dataset Columns

The Kaggle documentation describes these main variables:

- `airline`
- `flight`
- `source_city`
- `departure_time`
- `stops`
- `arrival_time`
- `destination_city`
- `class`
- `duration`
- `days_left`
- `price` — **target**

For the main model, `flight` is treated cautiously because it behaves like a high-cardinality flight-code identifier.


In [ ]:
# STEP 4 — Install packages
%pip install -q pandas numpy matplotlib seaborn scikit-learn joblib


In [ ]:
# STEP 5 — Imports
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries loaded successfully.")


## 6. Load the Kaggle Dataset

Download the CSV from the dataset page and place it in the same folder as this notebook.

The code searches common filenames automatically.


In [ ]:
DATA_PATH = None

candidate_files = [
    "Clean_Dataset.csv",
    "clean_dataset.csv",
    "flight_price.csv",
    "flight_prices.csv",
    "Data_Train.csv"
]

if DATA_PATH is None:
    matches = [f for f in candidate_files if os.path.exists(f)]
    if matches:
        DATA_PATH = matches[0]
    else:
        csv_files = glob.glob("*.csv")
        if csv_files:
            DATA_PATH = csv_files[0]

print("Selected file:", DATA_PATH)

if DATA_PATH is not None:
    df = pd.read_csv(DATA_PATH)
    print("Shape:", df.shape)
    display(df.head())
else:
    print("No CSV found. Download the Kaggle CSV and put it beside this notebook.")


## 7. Standardize Column Names


In [ ]:
if "df" in globals():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    print(df.columns.tolist())


## 8. Data Quality Check


In [ ]:
if "df" in globals():
    print("Shape:", df.shape)
    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))
    print("Duplicate rows:", df.duplicated().sum())
    print("\nTarget summary:")
    display(df["price"].describe())


## 9. Remove Non-Predictive Index

The dataset may contain an `index` column. It is an identifier rather than a meaningful explanatory variable.


In [ ]:
if "df" in globals() and "index" in df.columns:
    df = df.drop(columns=["index"])
    print("Index removed.")


## 10. Confirm This Is Regression

`price` is a continuous numerical target, so the task is supervised regression.

genui{"inference_regression_ml_learning_block":{"type_id":"LEAST_SQUARE_REGRESSION","locale_override":"en-IN"}}


In [ ]:
if "df" in globals():
    print("Target dtype:", df["price"].dtype)
    print("Unique price values:", df["price"].nunique())

    plt.figure(figsize=(9,5))
    sns.histplot(df["price"], bins=50, kde=True)
    plt.title("Distribution of Flight Prices")
    plt.xlabel("Price")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


## 11. EDA — Price by Airline


In [ ]:
if "df" in globals():
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x="airline", y="price")
    plt.xticks(rotation=30)
    plt.title("Flight Price by Airline")
    plt.tight_layout()
    plt.show()


## 12. EDA — Price by Class


In [ ]:
if "df" in globals():
    plt.figure(figsize=(7,5))
    sns.boxplot(data=df, x="class", y="price")
    plt.title("Flight Price by Class")
    plt.tight_layout()
    plt.show()


## 13. EDA — Days Left vs Price

`days_left` represents the number of days between booking and departure and is a key predictor in this dataset.


In [ ]:
if "df" in globals():
    sample = df.sample(min(10000, len(df)), random_state=42)

    plt.figure(figsize=(10,6))
    sns.scatterplot(data=sample, x="days_left", y="price", alpha=0.35)
    plt.title("Flight Price vs Days Left")
    plt.tight_layout()
    plt.show()


## 14. EDA — Duration vs Price


In [ ]:
if "df" in globals():
    plt.figure(figsize=(9,5))
    sns.scatterplot(data=sample, x="duration", y="price", alpha=0.35)
    plt.title("Flight Price vs Duration")
    plt.tight_layout()
    plt.show()


## 15. Numerical Correlation

Correlation helps describe linear association; it does not prove causality.


In [ ]:
if "df" in globals():
    numeric_cols = df.select_dtypes(include=np.number).columns

    plt.figure(figsize=(8,6))
    sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
    plt.title("Numerical Feature Correlation")
    plt.tight_layout()
    plt.show()


## 16. Feature Engineering

Create:
- `route`
- numerical representation of `stops`
- `duration_per_stop`

These features are constructed only from fields available before prediction.


In [ ]:
if "df" in globals():
    df["route"] = (
        df["source_city"].astype(str)
        + "_to_"
        + df["destination_city"].astype(str)
    )

    stop_map = {
        "zero": 0,
        "one": 1,
        "two_or_more": 2
    }

    df["stops_numeric"] = (
        df["stops"].astype(str).str.lower().map(stop_map)
    )

    df["duration_per_stop"] = (
        df["duration"] /
        (1 + df["stops_numeric"].fillna(0))
    )

    display(df.head())


## 17. Train / Test Split

We hold out 20% for final testing.

For this particular benchmark, a random split is reasonable because the dataset is not presented as a longitudinal panel of repeated price observations. For a real dynamic-fare system with chronological price observations, a time-based split should be used.


In [ ]:
if "df" in globals():
    X = df.drop(columns=["price"])
    y = df["price"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42
    )

    print("Training:", X_train.shape)
    print("Testing :", X_test.shape)


## 18. Preprocessing Pipeline

**Numerical:** median imputation + StandardScaler

**Categorical:** most-frequent imputation + OneHotEncoder

Putting preprocessing inside the pipeline helps prevent test-set leakage.


In [ ]:
if "df" in globals():
    numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    print("Numerical features:", numeric_features)
    print("Categorical features:", categorical_features)


# 19. Baseline — Linear Regression

Always establish a simple baseline before moving to complex models.


In [ ]:
if "df" in globals():
    linear_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", LinearRegression())
    ])

    linear_model.fit(X_train, y_train)
    linear_pred = linear_model.predict(X_test)

    print("Linear Regression R²:", r2_score(y_test, linear_pred))


# 20. Ridge and Lasso Regression

Regularization is useful when the encoded feature space contains correlated predictors.

- Ridge: L2 penalty
- Lasso: L1 penalty


In [ ]:
if "df" in globals():
    ridge_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", Ridge(alpha=10.0))
    ])

    lasso_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", Lasso(alpha=0.001, max_iter=20000))
    ])

    ridge_model.fit(X_train, y_train)
    lasso_model.fit(X_train, y_train)


# 21. Nonlinear Regression Models

Compare ensemble methods because flight pricing can contain nonlinear effects and interactions.


In [ ]:
if "df" in globals():
    rf_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ))
    ])

    gbr_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ])

    hgb_model = Pipeline([
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=42
        ))
    ])

    models = {
        "Linear Regression": linear_model,
        "Ridge": ridge_model,
        "Lasso": lasso_model,
        "Random Forest": rf_model,
        "Gradient Boosting": gbr_model,
        "HistGradientBoosting": hgb_model
    }

    predictions = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        predictions[name] = model.predict(X_test)

    print("All models trained.")


# 22. Regression Metrics

**MAE:** average absolute price error.

**RMSE:** gives more weight to large errors.

**R²:** proportion of target variance explained.

**MAPE:** percentage error; interpret cautiously because it can behave poorly near zero.


In [ ]:
if "predictions" in globals():
    rows = []

    for name, pred in predictions.items():
        mae = mean_absolute_error(y_test, pred)
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        r2 = r2_score(y_test, pred)

        nonzero = y_test != 0
        mape = np.mean(
            np.abs((y_test[nonzero] - pred[nonzero]) / y_test[nonzero])
        ) * 100

        rows.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "MAPE_%": mape
        })

    results_df = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    display(results_df)


# 23. Model Comparison


In [ ]:
if "results_df" in globals():
    plt.figure(figsize=(10,5))
    sns.barplot(data=results_df, x="RMSE", y="Model")
    plt.title("Model Comparison — RMSE")
    plt.tight_layout()
    plt.show()


# 24. 5-Fold Cross-Validation

Cross-validation checks whether performance is stable across multiple training/validation splits.


In [ ]:
if "df" in globals():
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    cv_rows = []

    for name in ["Linear Regression", "Ridge", "Random Forest", "Gradient Boosting"]:
        scores = cross_validate(
            models[name],
            X_train,
            y_train,
            cv=cv,
            scoring={
                "mae": "neg_mean_absolute_error",
                "rmse": "neg_root_mean_squared_error",
                "r2": "r2"
            },
            n_jobs=-1
        )

        cv_rows.append({
            "Model": name,
            "CV_MAE": -scores["test_mae"].mean(),
            "CV_RMSE": -scores["test_rmse"].mean(),
            "CV_R2": scores["test_r2"].mean()
        })

    cv_results = pd.DataFrame(cv_rows).sort_values("CV_RMSE")
    display(cv_results)


# 25. Hyperparameter Tuning — Random Forest

Tune the strongest candidate instead of blindly tuning every model.


In [ ]:
if "df" in globals():
    rf_pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
    ])

    param_dist = {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    }

    search = RandomizedSearchCV(
        rf_pipeline,
        param_distributions=param_dist,
        n_iter=10,
        scoring="neg_root_mean_squared_error",
        cv=3,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    print("Best parameters:")
    print(search.best_params_)
    print("Best CV RMSE:", -search.best_score_)


# 26. Final Test Evaluation


In [ ]:
if "search" in globals():
    final_model = search.best_estimator_
    final_pred = final_model.predict(X_test)

    final_metrics = {
        "MAE": mean_absolute_error(y_test, final_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, final_pred)),
        "R2": r2_score(y_test, final_pred)
    }

    print("FINAL TEST METRICS")
    for k, v in final_metrics.items():
        print(f"{k}: {v:.4f}")


# 27. Actual vs Predicted


In [ ]:
if "final_pred" in globals():
    plt.figure(figsize=(7,7))
    sns.scatterplot(x=y_test, y=final_pred, alpha=0.3)

    low = min(y_test.min(), final_pred.min())
    high = max(y_test.max(), final_pred.max())

    plt.plot([low, high], [low, high], linestyle="--")
    plt.xlabel("Actual Price")
    plt.ylabel("Predicted Price")
    plt.title("Actual vs Predicted Flight Fare")
    plt.tight_layout()
    plt.show()


# 28. Residual Diagnostics

Residual = Actual − Predicted.

Look for systematic patterns, large outliers and changing variance.


In [ ]:
if "final_pred" in globals():
    residuals = y_test.values - final_pred

    plt.figure(figsize=(10,5))
    sns.scatterplot(x=final_pred, y=residuals, alpha=0.3)
    plt.axhline(0, linestyle="--")
    plt.xlabel("Predicted Price")
    plt.ylabel("Residual")
    plt.title("Residuals vs Predicted Price")
    plt.tight_layout()
    plt.show()

    print("Residual mean:", residuals.mean())


# 29. Feature Importance

Feature importance helps answer:

> Which variables contribute most to the model's predictions?

Remember: feature importance is predictive association, not causal effect.


In [ ]:
if "search" in globals():
    fitted_preprocessor = final_model.named_steps["preprocess"]
    fitted_rf = final_model.named_steps["model"]

    feature_names = fitted_preprocessor.get_feature_names_out()
    importances = fitted_rf.feature_importances_

    importance_df = (
        pd.DataFrame({
            "Feature": feature_names,
            "Importance": importances
        })
        .sort_values("Importance", ascending=False)
        .head(20)
    )

    display(importance_df)

    plt.figure(figsize=(10,7))
    sns.barplot(data=importance_df, x="Importance", y="Feature")
    plt.title("Top Feature Importances")
    plt.tight_layout()
    plt.show()


# 30. New Flight Fare Prediction Function


In [ ]:
if "search" in globals():
    def predict_fare(
        airline,
        source_city,
        destination_city,
        departure_time,
        stops,
        arrival_time,
        flight_class,
        duration,
        days_left
    ):
        new_data = pd.DataFrame([{
            "airline": airline,
            "source_city": source_city,
            "destination_city": destination_city,
            "departure_time": departure_time,
            "stops": stops,
            "arrival_time": arrival_time,
            "class": flight_class,
            "duration": duration,
            "days_left": days_left,
            "route": f"{source_city}_to_{destination_city}",
            "stops_numeric": {
                "zero": 0,
                "one": 1,
                "two_or_more": 2
            }.get(str(stops).lower(), np.nan)
        }])

        new_data["duration_per_stop"] = (
            new_data["duration"] /
            (1 + new_data["stops_numeric"].fillna(0))
        )

        return float(final_model.predict(new_data)[0])

    print("Prediction function ready.")


# 31. Example Prediction

The example automatically uses a real category from the loaded dataset so that category mismatches are less likely.


In [ ]:
if "search" in globals():
    example_prediction = predict_fare(
        airline=df["airline"].iloc[0],
        source_city=df["source_city"].iloc[0],
        destination_city=df["destination_city"].iloc[0],
        departure_time=df["departure_time"].iloc[0],
        stops=df["stops"].iloc[0],
        arrival_time=df["arrival_time"].iloc[0],
        flight_class=df["class"].iloc[0],
        duration=float(df["duration"].median()),
        days_left=float(df["days_left"].median())
    )

    print("Predicted fare:", example_prediction)


# 32. Save the Model


In [ ]:
if "search" in globals():
    import joblib
    joblib.dump(final_model, "flight_fare_regression_model.pkl")
    print("Saved: flight_fare_regression_model.pkl")


# 33. Final Project Summary

### Problem
Predict flight ticket price from flight and booking characteristics.

### Type
**Supervised Regression**

### Target
`price`

### Main methods
- Linear Regression
- Ridge
- Lasso
- Random Forest Regressor
- Gradient Boosting Regressor
- HistGradientBoosting
- Randomized hyperparameter tuning

### Preprocessing
- missing-value handling
- one-hot encoding
- scaling
- feature engineering
- leakage-safe pipeline

### Evaluation
- MAE
- RMSE
- R²
- MAPE
- 5-fold cross-validation

### Interpretation
- feature importance
- actual-vs-predicted
- residual analysis

### Stronger future version
To make this a genuine dynamic-fare forecasting study, collect repeated prices for the same flight/route at multiple booking times and use chronological evaluation.


# 34. IIT Bombay Interview — Questions You Should Be Ready For

**Q1. Why regression?**  
Because the target is a continuous numerical fare.

**Q2. Why not accuracy?**  
Accuracy is not an appropriate primary metric for continuous regression.

**Q3. Why MAE and RMSE?**  
MAE gives an interpretable average error in price units; RMSE penalizes large errors more.

**Q4. Why start with Linear Regression?**  
It gives a simple baseline and provides a reference for judging whether nonlinear models add value.

**Q5. Why tree ensembles?**  
Fare relationships can be nonlinear and include interactions between route, class, stops and booking lead time.

**Q6. What is data leakage?**  
Using information during training that would not be available when making the prediction, or fitting transformations using the test set.

**Q7. Why use a Pipeline?**  
To keep preprocessing inside the training process and reduce leakage risk.

**Q8. Is `days_left` causal?**  
No. It is an observed predictor/booking-lead-time variable. Predictive importance should not be interpreted as a causal pricing effect.

**Q9. Why call it Dynamic Flight Fare Prediction carefully?**  
The available dataset supports fare prediction and analyzes booking lead time, but true dynamic pricing needs repeated observations over time for the same flight.

**Q10. What would you improve next?**  
Add chronological price observations, travel date, booking timestamp, holidays/events, route-level temporal features, and time-based validation.
